# Yoga Pose Detection - Master Classifier Inference

This notebook runs the trained master pose-classification model on a video file.
It discriminates between: **mountain**, **plank**, and **warrior2**.

It:
- Extracts MediaPipe landmarks from every Nth frame
- Engineers the exact same features used during training (from the master pose YAML config)
- Predicts the pose class + probability using the saved pipeline
- Displays the annotated video inline with skeleton overlay and prediction text

> Update only the paths in **Cell 1** - everything else is pose-agnostic.\n

---
## Cell 1 - Configuration (edit this cell only)\n

In [16]:
from pathlib import Path

#  Pose identity ─
POSE_NAME      = "master_pose"
EXPERIMENT_TAG = "master"   # update this to match the experiment tag used in master_model.ipynb

#  Saved artefact paths 
SAVED_FILES_ROOT = Path("../models/saved_files") / f"{POSE_NAME}_files"

DIR_MODELS   = SAVED_FILES_ROOT / "models"
DIR_METADATA = SAVED_FILES_ROOT / "metadata"
DIR_RESULTS  = SAVED_FILES_ROOT / "results"

#  Model selection ─
FORCE_MODEL: str | None = None
AVAILABLE_MODELS = ["logistic_regression", "svc", "knn", "xgboost"]

#  YAML config ─
YAML_PATH = Path("../../configs/poses/master_model.yaml")

#  Confidence threshold 
PROBABILITY_THRESHOLD = 0.0

#  Smoothing - sliding window majority vote 
SMOOTHING_WINDOW = 15   
MIN_HOLD_FRAMES  = 10   

#  Feature columns (must match the order used during training) 
FEATURE_COLUMNS = [
    "left_elbow_angle",
    "right_elbow_angle",
    "left_knee_angle",
    "right_knee_angle",
    "left_hip_angle",
    "right_hip_angle",
    "stance_width_normalized",
    "wrist_span_normalized",
    "torso_incline_angle",
    "leg_incline_angle",
    "left_arm_elevation",
    "right_arm_elevation",
    "left_hip_line_deviation",
    "right_hip_line_deviation",
]

#  Video 
# VIDEO_PATH = Path("test_videos/plank_demo.mp4")   
# VIDEO_PATH = Path("test_videos/Enhancer-Ultra HD-warrior2_test.mp4")   
VIDEO_PATH = Path("test_videos/Enhancer-HD-mountain_test.mp4")   
FRAME_STEP = 1    

#  Playback window ─
WINDOW_NAME  = f"{POSE_NAME} - Pose Detection"
WAITKEY_MS   = 1
DISPLAY_WIDTH: int | None = 960

#  MediaPipe model ─
MEDIAPIPE_MODEL_PATH = Path("pose_landmarker_lite.task")
MEDIAPIPE_MODEL_URL  = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
)

print("Configuration loaded.")
print(f"  Pose           : {POSE_NAME}")
print(f"  Saved files    : {SAVED_FILES_ROOT}  (exists={SAVED_FILES_ROOT.exists()})")
print(f"  YAML           : {YAML_PATH}  (exists={YAML_PATH.exists()})")
print(f"  Video          : {VIDEO_PATH}  (exists={VIDEO_PATH.exists()})")
print(f"  Force model    : {FORCE_MODEL or 'auto (best from results CSV)'}")


Configuration loaded.
  Pose           : master_pose
  Saved files    : ..\models\saved_files\master_pose_files  (exists=True)
  YAML           : ..\..\configs\poses\master_model.yaml  (exists=True)
  Video          : test_videos\Enhancer-HD-mountain_test.mp4  (exists=True)
  Force model    : auto (best from results CSV)


---
## Cell 2 - Imports\n

In [2]:
import base64
import math
import urllib.request
import warnings
import glob

import cv2
import joblib
import mediapipe as mp
import numpy as np
import pandas as pd
import yaml
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision import RunningMode

from IPython.display import display

print("All imports OK.")

All imports OK.


---
## Cell 3 - Skeleton & Colour Constants

The master model does not depend on orientation (unlike warrior2). All feature extraction is fixed for left/right.\n

In [3]:
# Full MediaPipe Pose skeleton connections (33 landmarks)
SKELETON_CONNECTIONS = [
    # Face
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10),
    # Torso
    (11, 12), (11, 23), (12, 24), (23, 24),
    # Left arm
    (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
    # Right arm
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
    # Left leg
    (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),
    # Right leg
    (24, 26), (26, 28), (28, 30), (28, 32), (30, 32),
]

# Feature-relevant connections derived from master_model.yaml.
# These will be highlighted in the drawn skeleton.
FEATURE_CONNECTIONS_FIXED = [
    (11, 13), (13, 15), # left elbow
    (12, 14), (14, 16), # right elbow
    (23, 25), (25, 27), # left knee
    (24, 26), (26, 28), # right knee
    (11, 23), (23, 25), # left hip
    (12, 24), (24, 26), # right hip
    (27, 28), # stance width
    (15, 16), # wrist span
    (11, 12), (23, 24), # torso incline / normalization
]

# BGR colours
COLOUR_SKELETON = (200, 200, 200)   # light grey  – standard skeleton
COLOUR_FEATURE  = (0, 220, 255)     # vivid cyan-yellow – feature-relevant
COLOUR_LANDMARK = (255, 255, 255)   # white dots for all landmarks
COLOUR_FEAT_DOT = (0, 220, 255)     # matching dot for feature landmarks

# Static landmark indices that are always feature-relevant
_STATIC_FEATURE_IDS = {11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28}

print(f"Standard skeleton connections : {len(SKELETON_CONNECTIONS)}")
print(f"Feature connections           : {len(FEATURE_CONNECTIONS_FIXED)}")
print(f"Feature landmark ids          : {sorted(_STATIC_FEATURE_IDS)}")

Standard skeleton connections : 35
Feature connections           : 16
Feature landmark ids          : [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]


---
## Cell 4 - Feature engineering helpers

These functions are a direct port of the training-time feature engineering pipeline (`master_model_fe.py`).
They operate on a single frame rather than a DataFrame so they are efficient for online inference.\n

In [4]:
EPSILON = 1e-6

#  Low-level coordinate helpers 

def _xy(landmarks, idx: int) -> np.ndarray:
    lm = landmarks[idx]
    return np.array([lm.x, lm.y], dtype=np.float64)

def _x(landmarks, idx: int) -> float:
    return float(landmarks[idx].x)

def _y(landmarks, idx: int) -> float:
    return float(landmarks[idx].y)

def _midpoint(landmarks, idx_a: int, idx_b: int) -> np.ndarray:
    return (_xy(landmarks, idx_a) + _xy(landmarks, idx_b)) / 2.0

#  Geometry primitives ─

def _angle_at_vertex_2d(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """2-D screen-plane angle at vertex B (X/Y only), in degrees."""
    ba = a - b
    bc = c - b
    cosine = np.clip(
        np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + EPSILON),
        -1.0, 1.0,
    )
    return float(np.degrees(np.arccos(cosine)))

def _euclidean_2d(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b))

def _point_to_line_2d(p: np.ndarray, a: np.ndarray, b: np.ndarray) -> float:
    """Perpendicular 2-D distance from point P to the line through A and B."""
    ab = b - a
    ap = p - a
    cross = abs(ab[0] * ap[1] - ab[1] * ap[0])
    return cross / (np.linalg.norm(ab) + EPSILON)

#  Per-feature-type compute functions 

def _compute_joint_angle(landmarks, cfg: dict) -> float:
    j = cfg["joints"]
    return _angle_at_vertex_2d(
        _xy(landmarks, j[0]),
        _xy(landmarks, j[1]),
        _xy(landmarks, j[2]),
    )

def _compute_spatial_distance(landmarks, cfg: dict) -> float:
    j    = cfg["joints"]
    dist = _euclidean_2d(_xy(landmarks, j[0]), _xy(landmarks, j[1]))

    if "normalization_factor" in cfg:
        nf    = cfg["normalization_factor"]
        scale = _euclidean_2d(_xy(landmarks, nf[0]), _xy(landmarks, nf[1]))
        dist  = dist / (scale + EPSILON)

    return dist

def _compute_alignment_offset(landmarks, cfg: dict) -> float:
    name    = cfg["name"]
    compute = cfg.get("compute", "").strip().lower()
    j       = cfg["joints"]

    try:
        if compute == "vertical_delta":
            val = abs(_y(landmarks, j[1]) - _y(landmarks, j[0]))

        elif compute == "perpendicular":
            val = _point_to_line_2d(
                _xy(landmarks, j[1]),
                _xy(landmarks, j[0]),
                _xy(landmarks, j[2]),
            )

        elif compute == "vector_incline_angle":
            if len(j) == 2:
                p0 = _xy(landmarks, j[0])
                p1 = _xy(landmarks, j[1])
            elif len(j) == 4:
                p0 = _midpoint(landmarks, j[0], j[1])
                p1 = _midpoint(landmarks, j[2], j[3])
            else:
                raise ValueError("vector_incline_angle requires 2 or 4 joints")
            
            delta = p1 - p0
            # arctan2(|dx|, |dy|) matches master_model_fe.py
            val = math.degrees(math.atan2(abs(delta[0]), abs(delta[1]) + EPSILON))

        else:
            raise ValueError(f"Alignment offset '{name}' has unsupported compute type '{compute}'.")

    except Exception as exc:
        warnings.warn(f"Failed computing alignment offset '{name}': {exc}")
        return float("nan")

    if "normalization_factor" in cfg:
        nf    = cfg["normalization_factor"]
        scale = _euclidean_2d(_xy(landmarks, nf[0]), _xy(landmarks, nf[1]))
        val   = val / (scale + EPSILON)

    return float(val)

#  Master per-frame feature extractor 

def extract_features_from_landmarks(
    landmarks,
    feat_config: dict,
    feature_columns: list[str],
) -> np.ndarray | None:
    feature_map: dict[str, float] = {}

    for cfg in feat_config.get("joint_angles", []):
        feature_map[cfg["name"]] = _compute_joint_angle(landmarks, cfg)

    for cfg in feat_config.get("spatial_distances", []):
        feature_map[cfg["name"]] = _compute_spatial_distance(landmarks, cfg)

    for cfg in feat_config.get("alignment_offsets", []):
        feature_map[cfg["name"]] = _compute_alignment_offset(landmarks, cfg)

    try:
        row = np.array([feature_map[col] for col in feature_columns], dtype=np.float64)
    except KeyError as e:
        raise KeyError(
            f"Feature {e} is in FEATURE_COLUMNS but was not computed. "
        ) from e

    if np.any(np.isnan(row)):
        return None

    return row

print("Feature engineering helpers defined.")

Feature engineering helpers defined.


---
## Cell 5 - Drawing helpers\n

In [5]:
def _landmark_px(lm, frame_w: int, frame_h: int) -> tuple[int, int]:
    return int(lm.x * frame_w), int(lm.y * frame_h)

def draw_skeleton(frame: np.ndarray, landmarks) -> np.ndarray:
    h, w = frame.shape[:2]
    px   = [_landmark_px(lm, w, h) for lm in landmarks]

    resolved_feat_conns = set()
    for a, b in FEATURE_CONNECTIONS_FIXED:
        resolved_feat_conns.add((a, b))
        resolved_feat_conns.add((b, a))

    # Draw standard skeleton (background)
    for a, b in SKELETON_CONNECTIONS:
        if (a, b) not in resolved_feat_conns:
            cv2.line(frame, px[a], px[b], COLOUR_SKELETON, 2, cv2.LINE_AA)

    # Draw feature-relevant connections (foreground)
    for a, b in FEATURE_CONNECTIONS_FIXED:
        cv2.line(frame, px[a], px[b], COLOUR_FEATURE, 3, cv2.LINE_AA)

    # Draw landmark dots
    for i, pos in enumerate(px):
        colour = COLOUR_FEAT_DOT if i in _STATIC_FEATURE_IDS else COLOUR_LANDMARK
        radius = 5 if i in _STATIC_FEATURE_IDS else 3
        cv2.circle(frame, pos, radius, colour, -1, cv2.LINE_AA)

    return frame

def draw_prediction_overlay(
    frame: np.ndarray,
    pose_name: str,
    predicted_class: str,
    probability: float | None,
    frame_idx: int,
) -> np.ndarray:
    font        = cv2.FONT_HERSHEY_SIMPLEX
    pad         = 10
    line_h      = 28
    small_scale = 0.55
    large_scale = 0.75

    prob_str = f"{probability * 100:.1f}%" if probability is not None else "N/A"

    banner_colour = (0, 130, 200)

    lines = [
        (f"Model : {pose_name.replace('_', ' ').title()}", small_scale, (220, 220, 220)),
        (f"Class : {predicted_class.replace('_', ' ').upper()}", large_scale, (255, 255, 255)),
        (f"Prob  : {prob_str}",                            small_scale, (200, 220, 255)),
        (f"Frame : {frame_idx}",                           small_scale, (180, 180, 180)),
    ]

    panel_h = pad * 2 + len(lines) * line_h
    panel_w = 360

    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (panel_w, panel_h), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

    cv2.rectangle(frame, (0, 0), (5, panel_h), banner_colour, -1)

    for row_i, (text, scale, colour) in enumerate(lines):
        y = pad + row_i * line_h + line_h // 2
        cv2.putText(frame, text, (14, y), font, scale, colour, 1, cv2.LINE_AA)

    return frame

print("Drawing helpers defined.")

Drawing helpers defined.


---
## Cell 6 - Load artefacts\n

In [6]:
#  1. Load YAML config ─
assert YAML_PATH.exists(), f"YAML not found: {YAML_PATH}"
with open(YAML_PATH, "r") as f:
    pose_yaml = yaml.safe_load(f)

feat_config = pose_yaml.get("features_config", {}).get("engineered_features", {})
assert feat_config, "No 'features_config.engineered_features' block found in YAML."

#  2. Select model ─
if FORCE_MODEL is not None:
    assert FORCE_MODEL in AVAILABLE_MODELS, (
        f"FORCE_MODEL='{FORCE_MODEL}' is not in {AVAILABLE_MODELS}"
    )
    selected_model = FORCE_MODEL
    print(f"Model forced to: {selected_model}")
else:
    # Find all CSVs in DIR_RESULTS
    csv_files = list(DIR_RESULTS.glob("*_experiment_results.csv"))
    if csv_files:
        results_csv = csv_files[0] # there should be just one
        results_df     = pd.read_csv(results_csv)
        # Using F1 macro for master pose
        best_row       = results_df.sort_values("test_f1_macro", ascending=False).iloc[0]
        selected_model = best_row["model"]
        print(f"Results CSV loaded. Best model by F1: {selected_model} "
              f"(test_f1={best_row['test_f1_macro']:.4f})")
    else:
        # Fallback if no results.csv
        selected_model = "xgboost"
        print(f"Results CSV not found, using fallback: {selected_model}")

#  3. Load pipeline and label encoder ─
# First try finding the right prefix by globbing DIR_MODELS
prefix = None
for f in DIR_MODELS.glob(f"*{selected_model}_pipeline.joblib"):
    prefix = f.name.replace("_pipeline.joblib", "")
    break

if prefix is None:
    prefix = f"{POSE_NAME}_{selected_model}" # default guess

pipeline_path = DIR_MODELS / f"{prefix}_pipeline.joblib"
encoder_path  = DIR_MODELS / f"{prefix}_label_encoder.joblib"

assert pipeline_path.exists(), f"Pipeline not found: {pipeline_path}"
assert encoder_path.exists(),  f"Label encoder not found: {encoder_path}"

pipeline      = joblib.load(pipeline_path)
label_encoder = joblib.load(encoder_path)

print(f"Pipeline loaded   : {pipeline_path.name}")
print(f"Label encoder     : {encoder_path.name}")
print(f"Known classes     : {list(label_encoder.classes_)}")

#  4. Check probability support ─
HAS_PROBA = hasattr(pipeline, "predict_proba")
print(f"Probability support: {HAS_PROBA}")

Results CSV loaded. Best model by F1: logistic_regression (test_f1=1.0000)
Pipeline loaded   : master_pose_master_3class_logistic_regression_pipeline.joblib
Label encoder     : master_pose_master_3class_logistic_regression_label_encoder.joblib
Known classes     : ['mountain', 'plank', 'warrior2']
Probability support: True


---
## Cell 7 - Ensure MediaPipe model file\n

In [8]:
def ensure_mediapipe_model(model_path: Path, url: str) -> None:
    if model_path.exists():
        print(f"MediaPipe model found: {model_path}")
        return
    print(f"Downloading MediaPipe model from:\n {url}")
    urllib.request.urlretrieve(url, model_path)
    print(f"Saved to: {model_path}")

ensure_mediapipe_model(MEDIAPIPE_MODEL_PATH, MEDIAPIPE_MODEL_URL)

MediaPipe model found: pose_landmarker_lite.task


---
## Cell 8 - Stream inference + display in a cv2 window\n

In [9]:
from collections import deque

class PredictionSmoother:
    def __init__(self, window_size: int, min_hold_frames: int):
        self.window        = deque(maxlen=window_size)
        self.min_hold      = min_hold_frames
        self.current_class = None
        self.hold_count    = 0

    def update(self, new_class: str) -> str:
        self.window.append(new_class)
        vote_winner = max(set(self.window), key=self.window.count)

        if self.current_class is None:
            self.current_class = vote_winner
            self.hold_count    = 1
        elif vote_winner == self.current_class:
            self.hold_count += 1
        elif self.hold_count >= self.min_hold:
            self.current_class = vote_winner
            self.hold_count    = 1
        else:
            self.hold_count += 1

        return self.current_class

In [17]:
def stream_inference_to_window(
    video_path: Path,
    mediapipe_model_path: Path,
    pipeline,
    label_encoder,
    feat_config: dict,
    feature_columns: list[str],
    pose_name: str,
    window_name: str,
    frame_step: int = 1,
    waitkey_ms: int = 1,
    display_width: int | None = None,
) -> dict:
    options = mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(
            model_asset_path=str(mediapipe_model_path)
        ),
        running_mode=RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
    )

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Cannot open video: {video_path}")
        return {}

    video_fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video   : {video_path.name}")
    print(f"FPS     : {video_fps:.1f}  |  Total frames : {total_frames}")
    print(f"Window  : '{window_name}'  - press Q or Esc to quit, Space to pause")

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    if display_width is not None:
        orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        scale  = display_width / max(orig_w, 1)
        cv2.resizeWindow(window_name, display_width, int(orig_h * scale))

    frame_count     = 0
    processed_count = 0
    no_pose_count   = 0
    pred_counts: dict[str, int] = {}
    paused  = False
    smoother = PredictionSmoother(SMOOTHING_WINDOW, MIN_HOLD_FRAMES)

    with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            if paused:
                key = cv2.waitKey(50) & 0xFF
                if key == ord(" "):
                    paused = False
                elif key in (ord("q"), 27):
                    print("Quit by user.")
                    break
                continue

            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            if (frame_count - 1) % frame_step != 0:
                continue

            processed_count += 1
            annotated    = frame.copy()
            timestamp_ms = int((frame_count / video_fps) * 1000)

            img_rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
            result   = landmarker.detect_for_video(mp_image, timestamp_ms)

            if not result.pose_landmarks:
                no_pose_count += 1
                cv2.putText(
                    annotated, "No pose detected",
                    (14, 50), cv2.FONT_HERSHEY_SIMPLEX,
                    1.0, (0, 60, 220), 2, cv2.LINE_AA,
                )
            else:
                landmarks = result.pose_landmarks[0]

                #  Skeleton ─
                draw_skeleton(annotated, landmarks)

                #  Feature engineering 
                feature_vec = extract_features_from_landmarks(
                    landmarks, feat_config, feature_columns
                )

                if feature_vec is None:
                    cv2.putText(
                        annotated, "Degenerate pose",
                        (14, 50), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (0, 100, 220), 2, cv2.LINE_AA,
                    )
                else:
                    #  Inference 
                    X_frame    = feature_vec.reshape(1, -1)
                    pred_enc   = pipeline.predict(X_frame)[0]
                    pred_class = label_encoder.inverse_transform([pred_enc])[0]

                    probability = None
                    if HAS_PROBA:
                        proba_vec   = pipeline.predict_proba(X_frame)[0]
                        probability = float(proba_vec[pred_enc])

                    # Apply confidence threshold
                    if probability is not None and probability < PROBABILITY_THRESHOLD:
                        raw_class = "unknown"
                    else:
                        raw_class = pred_class

                    # Smooth over sliding window
                    display_class    = smoother.update(raw_class)

                    pred_counts[display_class] = pred_counts.get(display_class, 0) + 1

                    draw_prediction_overlay(
                        annotated,
                        pose_name=pose_name,
                        predicted_class=display_class,
                        probability=probability,
                        frame_idx=frame_count,
                    )

            cv2.imshow(window_name, annotated)

            key = cv2.waitKey(waitkey_ms) & 0xFF
            if key in (ord("q"), 27):
                print("Quit by user.")
                break
            if key == ord(" "):
                paused = True

    cap.release()
    cv2.destroyWindow(window_name)

    inferrable = max(1, processed_count - no_pose_count)
    print(f"\nDone. Processed {processed_count} frames | No-pose: {no_pose_count}")
    print("Prediction distribution:")
    for cls, cnt in sorted(pred_counts.items(), key=lambda x: -x[1]):
        print(f"  {cls:<25} {cnt:>5} frames  ({cnt / inferrable * 100:.1f}%)")

    return {
        "processed_frames": processed_count,
        "no_pose_frames"  : no_pose_count,
        "pred_counts"     : pred_counts,
    }

if VIDEO_PATH.exists():
    summary = stream_inference_to_window(
        video_path           = VIDEO_PATH,
        mediapipe_model_path = MEDIAPIPE_MODEL_PATH,
        pipeline             = pipeline,
        label_encoder        = label_encoder,
        feat_config          = feat_config,
        feature_columns      = FEATURE_COLUMNS,
        pose_name            = POSE_NAME,
        window_name          = WINDOW_NAME,
        frame_step           = FRAME_STEP,
        waitkey_ms           = WAITKEY_MS,
        display_width        = DISPLAY_WIDTH,
    )
else:
    print(f"Could not run inference: {VIDEO_PATH} does not exist.")

Video   : Enhancer-HD-mountain_test.mp4
FPS     : 30.0  |  Total frames : 1801
Window  : 'master_pose - Pose Detection'  - press Q or Esc to quit, Space to pause

Done. Processed 1801 frames | No-pose: 0
Prediction distribution:
  mountain                   1801 frames  (100.0%)


---
## Cell 9 - Per-frame prediction summary (optional)\n

In [ ]:
def build_prediction_log(
    video_path: Path,
    mediapipe_model_path: Path,
    pipeline,
    label_encoder,
    feat_config: dict,
    feature_columns: list[str],
    frame_step: int = 1,
) -> pd.DataFrame:
    options = mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(
            model_asset_path=str(mediapipe_model_path)
        ),
        running_mode=RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
    )

    cap         = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return pd.DataFrame()

    fps         = cap.get(cv2.CAP_PROP_FPS) or 30.0
    rows        = []
    frame_count = 0

    with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            if (frame_count - 1) % frame_step != 0:
                continue

            timestamp_ms = int((frame_count / fps) * 1000)
            img_rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image     = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
            result       = landmarker.detect_for_video(mp_image, timestamp_ms)

            row = {"frame_number": frame_count}

            if not result.pose_landmarks:
                row["predicted_class"] = "no_pose"
                row["probability"]     = float("nan")
                rows.append(row)
                continue

            landmarks = result.pose_landmarks[0]
            feature_vec = extract_features_from_landmarks(
                landmarks, feat_config, feature_columns
            )

            if feature_vec is None:
                row["predicted_class"] = "degenerate_pose"
                row["probability"]     = float("nan")
                rows.append(row)
                continue

            for col, val in zip(feature_columns, feature_vec):
                row[col] = round(val, 6)

            X_frame    = feature_vec.reshape(1, -1)
            pred_enc   = pipeline.predict(X_frame)[0]
            pred_class = label_encoder.inverse_transform([pred_enc])[0]
            row["predicted_class"] = pred_class

            if HAS_PROBA:
                proba_vec          = pipeline.predict_proba(X_frame)[0]
                row["probability"] = round(float(proba_vec[pred_enc]), 4)
            else:
                row["probability"] = float("nan")

            rows.append(row)

    cap.release()
    return pd.DataFrame(rows)

if VIDEO_PATH.exists():
    log_df = build_prediction_log(
        video_path           = VIDEO_PATH,
        mediapipe_model_path = MEDIAPIPE_MODEL_PATH,
        pipeline             = pipeline,
        label_encoder        = label_encoder,
        feat_config          = feat_config,
        feature_columns      = FEATURE_COLUMNS,
        frame_step           = FRAME_STEP,
    )

    print(f"Per-frame log: {len(log_df)} rows")
    print("\nClass distribution across frames:")
    print(log_df["predicted_class"].value_counts().to_string())
    display(log_df.head(20))
